In [ ]:
%pip install -U transformers

In [ ]:
from pathlib import Path

INPUT_PATH = Path("./sentences.txt")
OUTPUT_PATH = Path("./clean_sentences.txt")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("bmd1905/vietnamese-correction-v2")
model = AutoModelForSeq2SeqLM.from_pretrained("bmd1905/vietnamese-correction-v2")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [ ]:
def correct_batch(sentences):
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            num_beams=1
        )

    texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    del inputs, outputs
    torch.cuda.empty_cache()

    return texts

def process_batch(batch, fout):
    if not batch:
        return

    indexes = []
    sentences = []

    for idx, sentence in batch:
        if sentence:
            indexes.append(idx)
            sentences.append(sentence)

    corrected_map = {}

    if sentences:
        corrected_sentences = correct_batch(sentences)

        for idx, corrected in zip(indexes, corrected_sentences):
            corrected_map[idx] = corrected

    for idx, sentence in batch:
        if sentence:
            fout.write(corrected_map[idx] + "\n")
        else:
            fout.write("\n")

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 128

with INPUT_PATH.open("r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)


with INPUT_PATH.open("r", encoding="utf-8") as fin, \
     OUTPUT_PATH.open("w", encoding="utf-8") as fout:

    batch = []

    for i, line in enumerate(tqdm(fin, total=total_lines, desc="Correcting")):
        sentence = line.strip()
        batch.append((i, sentence))

        if len(batch) >= BATCH_SIZE:
            process_batch(batch, fout)
            batch = []

    process_batch(batch, fout)

In [ ]:
lines = OUTPUT_PATH.read_text(encoding="utf-8").splitlines()
print("\n".join(lines[:100]))